# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore available record sets (each has a unique '@id')
record_sets = list(dataset.record_sets)
print(f"Record Sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}, description: {rs.description}")

# Display fields (columns) in each record set
for rs in record_sets:
    print(f"\nFields/Columns for RecordSet @id: {rs.id} - {rs.name}")
    for field in rs.fields:
        col_str = f"column: {field.column.id}" if hasattr(field, 'column') and field.column is not None else ""
        print(f"  Field @id: {field.id}, name: {field.name}, data type: {field.data_type} {col_str}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's get the list of record set @ids
record_set_ids = [rs.id for rs in record_sets]
print('Available RecordSet @ids:', record_set_ids)

# Example: Load all record sets into DataFrames
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet '@id' {rs.id}: {rs.name}")

# Show the columns (fields) of the first record set (update 'rs.id' to choose a specific one)
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
if chosen_record_set_id:
    print(f"\nColumns in record set {chosen_record_set_id}:\n", dataframes[chosen_record_set_id].columns.tolist())
    display(dataframes[chosen_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a RecordSet and numeric field by @id
chosen_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[chosen_record_set_id].copy() if chosen_record_set_id else pd.DataFrame()

# Hint: Replace with actual numeric field @id from overview above
print('DataFrame columns available:', df.columns.tolist())
# For this example, we'll search for a likely numeric field name
possible_numeric_fields = [col for col in df.columns if col.lower() in ['mw', 'molecular_weight', 'normalized_abundance', 'peptide_count', 'coverage']]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    numeric_field = df.columns[0] if len(df.columns) else None

if numeric_field is not None:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Try grouping by a likely string or category field
    possible_group_fields = [col for col in df.columns if 'desc' in col.lower() or 'accession' in col.lower() or 'sample' in col.lower() or 'group' in col.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Grouped data by {group_field}, mean of {numeric_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*This notebook demonstrates how to load, explore, and process a Croissant-standard dataset using the `mlcroissant` library.*

- We loaded dataset metadata and inspected available record sets and fields using their `@id`s.
- We extracted the data, filtered and normalized a selected numeric field, and visualized its distribution.
- The exact field identifiers and available analyses depend on the dataset's schema and content. You can adjust the chosen fields and visualizations as needed after inspecting the schema and initial data.

For more advanced usage, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/).